# CSCI 535 Practicum: Automatic Speech Recognition & Speaker Diarization
### CSCI-535 — Multimodal Probabilistic Learning of Human Communication

In this practicum we build and evaluate a full **speech understanding pipeline**: raw audio is transcribed with word-level timestamps by a Whisper ASR model, and a Pyannote-based diarization model segments the audio by speaker.  We then combine both outputs to produce a speaker-labeled transcript and evaluate quality using Word Error Rate (WER).

**Tools:** HuggingFace `transformers` · `pyannote.audio` · `jiwer` · `librosa`

> **Before you start:** In Colab go to **Runtime → Change runtime type → T4 GPU** and click Save.

## Learning Objectives

By the end of this practicum you will be able to:
1. Explain the components of a modern ASR system: mel spectrogram, encoder-decoder transformer, timestamp estimation.
2. Describe the speaker diarization pipeline: voice activity detection → speaker embeddings → clustering.
3. Transcribe audio using the HuggingFace **Whisper** pipeline with word-level timestamps.
4. Run **Pyannote** speaker diarization to identify who spoke when.
5. Combine ASR and diarization outputs to produce a speaker-labeled transcript.
6. Evaluate transcription quality with **Word Error Rate (WER)**.
7. Identify conditions that cause the pipeline to fail and explain why.

---
## Section 0 — Setup

In [ ]:
# 0.1 — Install dependencies
# The messages pip prints about "dependency conflicts" are EXPECTED on Colab.
# They are warnings about Colab's own packages (tensorflow, google-colab, gradio)
# and do NOT affect our code.  The cell below installs quietly and only
# surfaces genuine errors.

import subprocess, sys

def _pip(*packages):
    result = subprocess.run(
        [sys.executable, '-m', 'pip', 'install', '-q'] + list(packages),
        capture_output=True, text=True,
    )
    # Filter out resolver conflict warnings — show only real ERROR lines
    real_errors = [
        line for line in result.stderr.splitlines()
        if line.startswith('ERROR') and 'conflict' not in line.lower()
    ]
    if real_errors:
        print(''.join(real_errors))
    return result.returncode

# transformers / torch / torchaudio are pre-installed on Colab — do NOT reinstall.
_pip('pyannote.audio')     # speaker diarization (pulls pyannote.core automatically)
_pip('pyannote.metrics')   # DER evaluation (stretch exercise)
_pip('jiwer')              # Word Error Rate
_pip('librosa', 'soundfile', 'datasets')

# Quick smoke-test — if any import fails you will see a clear error here.
import pyannote.audio, pyannote.metrics, jiwer, librosa, soundfile, datasets
print('All packages imported successfully.')

In [ ]:
# 0.2 — Imports
import os, warnings
import numpy as np
import matplotlib.pyplot as plt
import librosa
import librosa.display
import soundfile as sf
import torch
import torchaudio
import jiwer
from transformers import pipeline as hf_pipeline
from pyannote.audio import Pipeline as DiarizePipeline
from pyannote.core import Annotation, Segment
from IPython.display import Audio, HTML, display
from datasets import load_dataset

warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-whitegrid')

In [ ]:
# 0.3 — Device check
device     = 'cuda' if torch.cuda.is_available() else 'cpu'
torch_dtype = torch.float16 if device == 'cuda' else torch.float32

print(f'Device     : {device}')
if device == 'cuda':
    print(f'GPU        : {torch.cuda.get_device_name(0)}')
    vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'VRAM       : {vram:.1f} GB')
    print(f'Torch dtype: {torch_dtype}')
else:
    print('WARNING: No GPU — inference will be slow. Enable GPU in Runtime → Change runtime type.')

In [ ]:
# 0.4 — HuggingFace token
# Pyannote diarization models are gated.  To obtain a token:
#   1. Create a free account at huggingface.co
#   2. Accept the licence at hf.co/pyannote/speaker-diarization-3.1
#   3. Go to hf.co/settings/tokens and create a READ token
#   4. Paste it below.

HF_TOKEN = 'hf_YOUR_TOKEN_HERE'   # <-- replace this

assert HF_TOKEN != 'hf_YOUR_TOKEN_HERE', \
    'Replace HF_TOKEN with your actual token before running Sections 5-6.'

---
## Section 1 — Background

Read through this section carefully before proceeding to the exercises.

### 1.1 Automatic Speech Recognition

Modern ASR systems like **Whisper** (OpenAI, 2022) are trained on 680,000 hours of multilingual speech.  The architecture is an encoder-decoder transformer:

```
Raw PCM audio
      │
      ▼
┌──────────────────────────────────────┐
│  Log Mel Spectrogram (80 filterbanks)│  25 ms window, 10 ms hop at 16 kHz
└──────────────────────────────────────┘
      │
      ▼
┌──────────────────────────────────────┐
│  Conv stem (2 layers)                │
└──────────────────────────────────────┘
      │
      ▼
┌──────────────────────────────────────┐
│  Transformer Encoder  (N blocks)     │  audio → latent representation
└──────────────────────────────────────┘
      │
      ▼
┌──────────────────────────────────────┐
│  Transformer Decoder  (N blocks)     │  autoregressively generates tokens
└──────────────────────────────────────┘
      │
      ▼
Text tokens + attention-based timestamps
```

In this practicum we use the HuggingFace `transformers` `pipeline` API, which handles chunking, batching, and timestamp extraction automatically.

| Model | Params | VRAM | Speed |
|---|---|---|---|
| `whisper-tiny` | 39 M | ~0.5 GB | Very fast |
| `whisper-base` | 74 M | ~0.7 GB | Fast |
| `whisper-large-v3-turbo` | 809 M | ~2 GB | **Recommended** |
| `whisper-large-v3` | 1 550 M | ~4.7 GB | Highest accuracy |

`whisper-large-v3-turbo` achieves near-large-v3 accuracy at 8× the speed — the **recommended default** for this practicum.

### 1.2 Speaker Diarization

Diarization answers *"who spoke when?"*  The Pyannote 3.1 pipeline has three stages:

```
Audio
  │
  ▼
┌─────────────────────────────────────┐
│  Voice Activity Detection  (VAD)    │  SincNet + LSTM, frame-level labels
└─────────────────────────────────────┘
  │
  ▼
┌─────────────────────────────────────┐
│  Speaker Embedding                  │  ECAPA-TDNN → 192-d d-vector / segment
└─────────────────────────────────────┘
  │
  ▼
┌─────────────────────────────────────┐
│  Agglomerative Clustering           │  merges segments by cosine similarity
└─────────────────────────────────────┘
  │
  ▼
pyannote.core.Annotation:
  [0.0–2.3 s]  SPEAKER_00
  [2.3–5.1 s]  SPEAKER_01
  ...
```

Key hyperparameters:
- `min_speakers` / `max_speakers` — hard constraints on cluster count.  Set these when you know the number of speakers.

### 1.3 Word-Level Timestamps and Speaker Assignment

Whisper outputs **chunk-level** timestamps — start/end of each 30-second processing window.  Requesting `return_timestamps="word"` from the HuggingFace pipeline adds finer **word-level** timestamps estimated from the decoder's cross-attention weights.

Once we have both word timestamps and diarization segments, assigning speakers to words is a simple overlap problem:

```
Diarization output:
  [0.0 – 3.2 s]  SPEAKER_00
  [3.8 – 7.1 s]  SPEAKER_01

Word timestamps:
  [0.2–0.5]  "Hello"   → midpoint 0.35 s → SPEAKER_00
  [0.6–0.9]  "there"   → midpoint 0.75 s → SPEAKER_00
  [3.9–4.2]  "Hi"      → midpoint 4.05 s → SPEAKER_01
```

For each word, we take the midpoint of its timestamp and look up which speaker's segment contains that time.

---
## Section 2 — Audio Loading and Visualization

We use two audio sources, both loaded from the same `librispeech_asr` dataset — no external downloads needed:

| Source | Used in | Why |
|---|---|---|
| **Single-speaker** (clean, known transcript) | Sections 3–4, Section 7 | Reproducible WER evaluation |
| **2-speaker mix** (two speakers interleaved, exact ground truth) | Sections 5–6, DER stretch | Diarization with known boundaries |

In [ ]:
# 2a — Single-speaker audio from LibriSpeech test-clean
print('Loading LibriSpeech test-clean sample ...')
libri = load_dataset(
    'librispeech_asr', 'clean', split='test',
    streaming=True, trust_remote_code=True,
)
sample = next(iter(libri))

SINGLE_AUDIO   = sample['audio']['array'].astype(np.float32)
SINGLE_SR      = sample['audio']['sampling_rate']   # 16000 Hz
REFERENCE_TEXT = sample['text'].lower()

print(f'Sample rate : {SINGLE_SR} Hz')
print(f'Duration    : {len(SINGLE_AUDIO) / SINGLE_SR:.2f} s')
print(f'Reference   : {REFERENCE_TEXT}')

In [ ]:
# 2b — Build a 2-speaker sample from LibriSpeech (no YouTube needed)
TARGET_SPEAKERS  = 2
UTTS_PER_SPEAKER = 4
SILENCE_GAP_S    = 0.6

print('Building 2-speaker audio from LibriSpeech test-clean ...')

libri_stream = load_dataset(
    'librispeech_asr', 'clean', split='test',
    streaming=True, trust_remote_code=True,
)

speaker_utts = {}
for sample in libri_stream:
    spk = str(sample['speaker_id'])
    if spk not in speaker_utts:
        speaker_utts[spk] = []
    if len(speaker_utts[spk]) < UTTS_PER_SPEAKER:
        speaker_utts[spk].append({
            'audio': sample['audio']['array'].astype(np.float32),
            'text':  sample['text'].lower(),
        })
    if (len(speaker_utts) >= TARGET_SPEAKERS and
            all(len(v) == UTTS_PER_SPEAKER for v in speaker_utts.values())):
        break

speaker_ids = list(speaker_utts.keys())[:TARGET_SPEAKERS]
silence     = np.zeros(int(SILENCE_GAP_S * 16000), dtype=np.float32)

segments_audio = []
GROUND_TRUTH   = []
cursor_s       = 0.0

for turn_idx in range(UTTS_PER_SPEAKER):
    for spk_idx, spk_id in enumerate(speaker_ids):
        utt = speaker_utts[spk_id][turn_idx]
        dur = len(utt['audio']) / 16000
        GROUND_TRUTH.append({
            'speaker': f'SPEAKER_{spk_idx:02d}',
            'text':    utt['text'],
            'start':   cursor_s,
            'end':     cursor_s + dur,
        })
        segments_audio.append(utt['audio'])
        segments_audio.append(silence)
        cursor_s += dur + SILENCE_GAP_S

MULTI_AUDIO = np.concatenate(segments_audio)
MULTI_SR    = 16000
MULTI_PATH  = 'multi_speaker.wav'
sf.write(MULTI_PATH, MULTI_AUDIO, MULTI_SR)

total_dur = len(MULTI_AUDIO) / MULTI_SR
print(f'Speakers : {len(speaker_ids)}  (IDs: {speaker_ids})')
print(f'Turns    : {len(GROUND_TRUTH)}')
print(f'Duration : {total_dur:.1f} s  ({total_dur/60:.1f} min)')
print(f'Saved    : {MULTI_PATH}')
print()
print(f'  {"Speaker":<12}  {"Start":>6}  {"End":>6}  Text')
print('  ' + '-'*70)
for gt in GROUND_TRUTH:
    print(f'  {gt["speaker"]:<12}  {gt["start"]:>6.1f}  {gt["end"]:>6.1f}  {gt["text"][:50]}')

In [ ]:
# Listen to both audio clips before proceeding
print('Single speaker (LibriSpeech):')
display(Audio(SINGLE_AUDIO, rate=SINGLE_SR))

print('2-speaker mix (interleaved turns):')
display(Audio(MULTI_AUDIO, rate=MULTI_SR))

print(f'Reference transcript:  {REFERENCE_TEXT}')

### 2.1 — Waveform and Spectrogram Visualization

Run the two cells below to visualize the audio signals.

In [ ]:
def plot_waveform(audio_array: np.ndarray, sample_rate: int, title: str = 'Waveform') -> None:
    time = np.linspace(0, len(audio_array) / sample_rate, len(audio_array))
    fig, ax = plt.subplots(figsize=(12, 2.5))
    ax.plot(time, audio_array, linewidth=0.4, color='steelblue')
    ax.set_xlabel('Time (s)')
    ax.set_ylabel('Amplitude')
    ax.set_xlim(0, time[-1])
    ax.set_title(title)
    plt.tight_layout(); plt.show()

In [ ]:
plot_waveform(SINGLE_AUDIO, SINGLE_SR, 'Single Speaker — Waveform')
plot_waveform(MULTI_AUDIO,  MULTI_SR,  '2-Speaker Mix — Waveform')

### 2.2 — Mel Spectrogram

Whisper uses **80 mel filterbanks**, a 25 ms window, and a 10 ms hop — matching the parameters below.

In [ ]:
def plot_mel_spectrogram(
    audio_array : np.ndarray,
    sample_rate : int,
    n_mels      : int = 80,
    n_fft       : int = 400,
    hop_length  : int = 160,
    title       : str = 'Log Mel Spectrogram',
) -> None:
    S    = librosa.feature.melspectrogram(
               y=audio_array, sr=sample_rate,
               n_mels=n_mels, n_fft=n_fft, hop_length=hop_length)
    S_db = librosa.power_to_db(S, ref=np.max)
    fig, ax = plt.subplots(figsize=(12, 3))
    img = librosa.display.specshow(
              S_db, sr=sample_rate, hop_length=hop_length,
              x_axis='time', y_axis='mel', ax=ax)
    fig.colorbar(img, ax=ax, label='dB')
    ax.set_title(title)
    plt.tight_layout(); plt.show()

In [ ]:
plot_mel_spectrogram(SINGLE_AUDIO, SINGLE_SR, title='Single Speaker — Log Mel Spectrogram (80 bins)')


> **Q2.1** Look at the mel spectrogram of the audio clip.  At what frequency range is most speech energy concentrated? Can you see pauses? What about vowels?


---
## Section 3 — Automatic Speech Recognition

We use the HuggingFace `transformers` pipeline to run Whisper.  The pipeline handles audio chunking (Whisper processes 30-second windows), batching, and timestamp extraction automatically.

The default model is `openai/whisper-large-v3-turbo` — 8× faster than `large-v3` with near-identical accuracy.

### Exercise 3.1 — Load the ASR Pipeline

Use `load_asr_pipeline` to instantiate a Whisper pipeline.

In [ ]:
#DEFAULT_MODEL = 'openai/whisper-large-v3-turbo'
DEFAULT_MODEL = 'openai/whisper-base'

def load_asr_pipeline(model_id: str = DEFAULT_MODEL,
                      device: str = 'cuda',
                      torch_dtype = torch.float16):
    '''
    Load a Whisper model via the HuggingFace transformers pipeline.

    Args:
        model_id    : HuggingFace model ID, e.g. "openai/whisper-large-v3-turbo",
                      "openai/whisper-base", "openai/whisper-tiny".
        device      : "cuda" or "cpu".
        torch_dtype : torch.float16 (GPU) or torch.float32 (CPU).

    Returns:
        HuggingFace ASR pipeline ready for inference.

    '''
    return hf_pipeline(
         "automatic-speech-recognition",
         model=model_id,
         torch_dtype=torch_dtype,
         device=device,
     )

asr_pipe = load_asr_pipeline(DEFAULT_MODEL, device, torch_dtype)
print(f'Loaded: {DEFAULT_MODEL}')

### Exercise 3.2 — Transcribe with Segment Timestamps

Completing `transcribe`.  Start with **segment-level** timestamps (`return_timestamps=True`) to see the coarse output; we will add word-level timestamps in Exercise 3.3.

In [ ]:
def transcribe(asr_pipe, audio_array: np.ndarray, sample_rate: int,
               chunk_length_s: int = 30, batch_size: int = 8,
               return_timestamps = True) -> dict:
    '''
    Transcribe audio using a HuggingFace Whisper pipeline.

    Args:
        asr_pipe        : Pipeline from load_asr_pipeline().
        audio_array     : 1-D float32 numpy array at sample_rate Hz.
        sample_rate     : Must be 16000 for Whisper.
        chunk_length_s  : Audio chunk size (30 s = Whisper context window).
        batch_size      : Chunks processed in parallel (reduce if OOM).
        return_timestamps: True for chunk-level, "word" for word-level.

    Returns:
        dict with:
            "text"   : full transcript string
            "chunks" : list of dicts, each with "text" and "timestamp" (start, end).
    '''
    return asr_pipe(
         {"array": audio_array, "sampling_rate": sample_rate},
         chunk_length_s=chunk_length_s,
         batch_size=batch_size,
         return_timestamps=return_timestamps,
         ignore_warning=True
     )

seg_result = transcribe(asr_pipe, SINGLE_AUDIO, SINGLE_SR, return_timestamps=True)

print(f'Full transcript: {seg_result["text"]}')
print()
print(f'  {"Start":>6}  {"End":>6}  Text')
print('  ' + '-'*60)
for chunk in seg_result['chunks']:
    s, e = chunk['timestamp']
    print(f'  {s:>6.2f}  {e:>6.2f}  {chunk["text"].strip()}')

> **Q3.1** How would you deal with audio longer than 30 seconds?
### Exercise 3.3 — Word-Level Timestamps

Re-run `transcribe` with `return_timestamps="word"` to get per-word timing.

In [ ]:
word_result = transcribe(asr_pipe, SINGLE_AUDIO, SINGLE_SR, return_timestamps='word')

print(f'Words found: {len(word_result["chunks"])}')
print()
print(f'  {"Word":<22}  {"Start":>6}  {"End":>6}')
print('  ' + '-'*40)
for chunk in word_result['chunks'][:25]:
    word = chunk['text'].strip()
    s, e = chunk['timestamp']
    print(f'  {word:<22}  {s if s is not None else float("nan"):>6.3f}  {e if e is not None else float("nan"):>6.3f}')

> **Q3.2** Compare the segment-level timestamps (Exercise 3.2) with the word-level timestamps (Exercise 3.3).  How does the segment boundary compare to the word boundaries within that segment?  If a speaker change happened at second 2.1 inside a segment that spans 0.0–4.8 s, would segment-level timestamps be sufficient to assign the correct speaker label?

---
## Section 4 — Visualizing Timestamps

Before diving into diarization, we visualize the two levels of timestamps to build intuition for why word-level precision matters.

### Exercise 4.1 — Timestamp Comparison Plot

Implementing `plot_timestamp_comparison` to display segment-level and word-level timestamps side by side on the same time axis.

In [ ]:
def plot_timestamp_comparison(seg_result: dict, word_result: dict,
                              title: str = 'Timestamp Comparison') -> None:
    '''
    Plot segment-level and word-level timestamps on two aligned rows.

    Args:
        seg_result  : Output of transcribe(..., return_timestamps=True).
        word_result : Output of transcribe(..., return_timestamps="word").
        title       : Overall figure title.

    Steps:
        1. Create a figure with 2 subplots sharing the x-axis.
        2. Top row: draw one horizontal bar per chunk in seg_result["chunks"],
           spanning [start, end].  Add the truncated text as a label.
        3. Bottom row: draw one bar per word in word_result["chunks"].
        4. Label y-axes ("Segments", "Words"), x-axis ("Time (s)"), add title.

    '''
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 4), sharex=True)

    for chunk in seg_result["chunks"]:
         s, e = chunk["timestamp"]
         if s is None or e is None: continue
         ax1.barh(0, e - s, left=s, height=0.6, color="steelblue",
                  edgecolor="white", linewidth=0.5)
         ax1.text((s + e) / 2, 0, chunk["text"].strip(),
                  ha="center", va="center", fontsize=12, color="white", clip_on=True)
    ax1.set_yticks([0]); ax1.set_yticklabels(["Segments"])
    ax1.set_title("Chunk-level timestamps")
    for chunk in word_result["chunks"]:
        s, e = chunk["timestamp"]
        if s is None or e is None: continue
        ax2.barh(0, e - s, left=s, height=0.6, color="darkorange",
                 edgecolor="white", linewidth=0.3)
        #write words one by one
        ax2.text((s + e) / 2, 0, chunk["text"].strip(),
                 ha="center", va="center", fontsize=12, color="white", clip_on=True)

    ax2.set_yticks([0]); ax2.set_yticklabels(["Words"])
    ax2.set_title("Word-level timestamps")

    ax2.set_xlabel('Time (s)')
    fig.suptitle(title)
    plt.tight_layout(); plt.show()


plot_timestamp_comparison(seg_result, word_result,
                          'Segment-level vs. Word-level Timestamps')

> **Q4.1** Do these two alignments agree?  By how much do they differ (in milliseconds)?  Why does this level of precision matter for speaker diarization or subtitle generation?

---
## Section 5 — Speaker Diarization

We now switch to `MULTI_AUDIO` — the 2-speaker mix built in Section 2.  Diarization partitions the audio into time segments and assigns a speaker label to each.

### Exercise 5.1 — Load the Diarization Pipeline

`load_diarization_pipeline` to load the Pyannote model.

In [ ]:
SPEAKER_COLORS = ['#4C72B0', '#DD8452', '#55A868', '#C44E52', '#8172B2']

def load_diarization_pipeline(hf_token: str, device: str):
    '''
    Load the Pyannote 3.1 speaker diarization pipeline.

    Args:
        hf_token : HuggingFace READ token (for the gated Pyannote model).
        device   : "cuda" or "cpu".

    Returns:
        Pyannote Pipeline moved to the target device.

    '''
    pipe = DiarizePipeline.from_pretrained(
         "pyannote/speaker-diarization-3.1",
         token=hf_token,
     )
    return pipe.to(torch.device(device))

diarize_pipe = load_diarization_pipeline(HF_TOKEN, device)
print('Diarization pipeline loaded.')

### Exercise 5.2 — Run Diarization

Complete `run_diarization`.  Pyannote accepts either a file path or a dict with `waveform` (torch tensor) and `sample_rate`.

In [ ]:
def run_diarization(diarize_pipe, audio_array: np.ndarray, sample_rate: int,
                    min_speakers: int = None, max_speakers: int = None):
    '''
    Run Pyannote speaker diarization on a numpy audio array.

    Args:
        diarize_pipe  : Pipeline from load_diarization_pipeline().
        audio_array   : 1-D float32 numpy array at sample_rate Hz.
        sample_rate   : Must be 16000.
        min_speakers  : Minimum number of speakers (None = auto).
        max_speakers  : Maximum number of speakers (None = auto).

    Returns:
        pyannote.core.Annotation — iterable of (Segment, track, speaker_label).


    '''
    waveform = torch.from_numpy(audio_array).unsqueeze(0)
    return diarize_pipe(
         {"waveform": waveform, "sample_rate": sample_rate},
         min_speakers=min_speakers,
         max_speakers=max_speakers,
     )

diarization = run_diarization(diarize_pipe, MULTI_AUDIO, MULTI_SR,
                               min_speakers=2, max_speakers=2)

annotation = diarization.speaker_diarization
print(f'Segments detected: {len(annotation)}')
print()
print(f'  {"Speaker":<14}  {"Start":>6}  {"End":>6}')
print('  ' + '-'*34)
for segment, _, speaker in annotation.itertracks(yield_label=True):
    print(f'  {speaker:<14}  {segment.start:>6.2f}  {segment.end:>6.2f}')

### Exercise 5.3 — Speaker Timeline Plot

Implementing `plot_speaker_timeline` to visualize diarization output as a Gantt chart.

In [ ]:
def plot_speaker_timeline(diarization, ground_truth: list = None,
                          title: str = 'Speaker Timeline') -> None:
    '''
    Plot a Gantt-style speaker diarization timeline.

    Args:
        diarization  : pyannote.core.Annotation from run_diarization().
        ground_truth : Optional list of GT dicts with start, end, speaker keys.
                       If provided, draws a second row showing the ground truth.
        title        : Plot title.

    Steps:
        1. Collect unique speaker labels from diarization.labels().
        2. For each (segment, _, speaker) in diarization.itertracks(yield_label=True):
               ax.barh(y_pos, segment.end - segment.start, left=segment.start, ...)
        3. If ground_truth provided, draw a second axis below with GT bars.
        4. Color-code by speaker using SPEAKER_COLORS.

    Hint:
        for segment, _, speaker in diarization.itertracks(yield_label=True):
            start, end = segment.start, segment.end
    '''
    n_rows = 2 if ground_truth else 1
    fig, axes = plt.subplots(n_rows, 1, figsize=(14, 2 * n_rows), sharex=True)
    if n_rows == 1:
        axes = [axes]

    speakers   = sorted(diarization.labels())
    spk_to_y   = {spk: i for i, spk in enumerate(speakers)}
    spk_to_col = {spk: SPEAKER_COLORS[i % len(SPEAKER_COLORS)]
        for i, spk in enumerate(speakers)}

    for seg, _, spk in diarization.itertracks(yield_label=True):
         axes[0].barh(spk_to_y[spk], seg.end - seg.start, left=seg.start,
                     height=0.6, color=spk_to_col[spk],
                      edgecolor="white", linewidth=0.5)
    axes[0].set_yticks(list(spk_to_y.values()))
    axes[0].set_yticklabels(list(spk_to_y.keys()))
    axes[0].set_title("System diarization output")
    #
    if ground_truth:
        gt_spks   = sorted(set(g["speaker"] for g in ground_truth))
        gt_to_y   = {s: i for i, s in enumerate(gt_spks)}
        gt_to_col = {s: SPEAKER_COLORS[i % len(SPEAKER_COLORS)]
                     for i, s in enumerate(gt_spks)}
        for g in ground_truth:
            axes[1].barh(gt_to_y[g["speaker"]], g["end"] - g["start"],
                         left=g["start"], height=0.6,
                         color=gt_to_col[g["speaker"]],
                         edgecolor="white", linewidth=0.5)
        axes[1].set_yticks(list(gt_to_y.values()))
        axes[1].set_yticklabels(list(gt_to_y.keys()))
        axes[1].set_title("Ground truth")

    axes[-1].set_xlabel('Time (s)')
    fig.suptitle(title)
    plt.tight_layout(); plt.show()


plot_speaker_timeline(annotation, ground_truth=GROUND_TRUTH,
                      title='Multi-Speaker Audio — Diarization vs. Ground Truth')
display(Audio(MULTI_AUDIO, rate=MULTI_SR))


> **Q5.1** Compare the system's diarization output (top row) to the ground truth (bottom row).  Are speaker boundaries detected accurately?  Are there gaps between turns that are assigned to the wrong speaker?  How does the Pyannote pipeline handle the silence gaps between utterances?

---
## Section 6 — Full Pipeline: Who Said What?

We now combine word-level ASR timestamps with diarization segments to assign a speaker label to each transcribed word.  You will implement the speaker-word matching logic yourself.

### Exercise 6.1 — Assign Speakers to Words

Completing `assign_speakers_to_words`.  For each word, find which diarization segment contains the word's midpoint and assign that speaker's label.

In [ ]:
def assign_speakers_to_words(word_chunks: list, diarization) -> list:
    '''
    Assign a speaker label to each transcribed word based on timestamp overlap.

    Args:
        word_chunks  : list of dicts from transcribe(..., return_timestamps="word"),
                       each with keys "text" and "timestamp" (start, end).
        diarization  : pyannote.core.Annotation from run_diarization().

    Returns:
        list of dicts:
            {"word": str, "start": float, "end": float, "speaker": str}

    Algorithm:
        For each word:
          1. Extract start and end from chunk["timestamp"].
             Skip words where either is None.
          2. Compute midpoint = (start + end) / 2.
          3. Iterate over diarization.itertracks(yield_label=True).
             Find the segment whose [start, end] interval contains midpoint.
          4. Assign that segment\'s speaker label (or "UNKNOWN" if none found).

                ...
    '''
    labeled_words = []

    for chunk in word_chunks:
        word = chunk['text'].strip()
        start, end = chunk['timestamp']

        # -- YOUR CODE HERE --------------------------------------------------
        if start is None or end is None:
            continue
        midpoint = (start + end) / 2
        speaker  = "UNKNOWN"
        for segment, _, label in diarization.itertracks(yield_label=True):
            if segment.start <= midpoint <= segment.end:
                speaker = label
                break
        labeled_words.append({
            "word": word, "start": start, "end": end, "speaker": speaker,
        })

        # ─────────────────────────────────────────────────────────────────

    return labeled_words


# Run full pipeline on multi-speaker audio
print('Transcribing multi-speaker audio ...')
multi_word_result = transcribe(asr_pipe, MULTI_AUDIO, MULTI_SR, return_timestamps='word')

print('Assigning speakers ...')
labeled_words = assign_speakers_to_words(multi_word_result['chunks'], annotation)

print(f'Labeled {len(labeled_words)} words.')

In [ ]:
# Print speaker-labeled transcript
current_speaker = None
line = ''
for w in labeled_words:
    if w['speaker'] != current_speaker:
        if line:
            print(f'  {line.strip()}')
        current_speaker = w['speaker']
        idx = int(w['speaker'].split('_')[-1]) if '_' in w['speaker'] else 0
        print(f'\n[{w["speaker"]}  @ {w["start"]:.1f}s]')
        line = ''
    line += w['word'] + ' '
if line:
    print(f'  {line.strip()}')

In [ ]:
# Pre-filled: color-coded HTML transcript
def make_html_transcript(labeled_words: list) -> str:
    lines = [
        '<style>',
        '  .spk { font-weight:bold; margin-top:8px; display:block }',
        '  .word { border-radius:2px; padding:0 2px }',
        '</style>',
        '<div style="font-family:monospace; font-size:14px; line-height:2">',
    ]
    current_speaker = None
    for w in labeled_words:
        if w['speaker'] != current_speaker:
            current_speaker = w['speaker']
            idx = int(w['speaker'].split('_')[-1]) if '_' in w['speaker'] else 0
            c   = SPEAKER_COLORS[idx % len(SPEAKER_COLORS)]
            lines.append(
                f'<span class="spk" style="color:{c}">'
                f'[{w["speaker"]} @ {w["start"]:.1f}s]</span>'
            )
        lines.append(f'<span class="word">{w["word"]} </span>')
    lines.append('</div>')
    return '\n'.join(lines)

display(HTML(make_html_transcript(labeled_words)))

---
## Section 7 — Evaluation

### 7.1 Word Error Rate

$$\text{WER} = \frac{S + D + I}{N}$$

| Symbol | Meaning |
|---|---|
| $S$ | Substitutions — wrong word |
| $D$ | Deletions — missing word |
| $I$ | Insertions — extra word |
| $N$ | Total words in the reference |

WER can exceed 1.0 when there are many insertions.  State-of-the-art English ASR achieves ~2–4 % WER on clean speech.

### Exercise 7.1 — Word Error Rate

Implementing `compute_wer` and `compute_detailed_errors`.

In [ ]:
def compute_wer(reference: str, hypothesis: str) -> float:
    '''
    Compute Word Error Rate.

    Args:
        reference  : Ground-truth transcript (lowercase, no punctuation).
        hypothesis : ASR output transcript.

    Returns:
        WER as a float (0.0 = perfect).

    '''
    return jiwer.wer(reference, hypothesis)


def compute_detailed_errors(reference: str, hypothesis: str):
    '''
    Return a detailed error breakdown (substitutions, deletions, insertions).

    Args:
        reference  : Ground-truth transcript.
        hypothesis : ASR output transcript.

    Returns:
        jiwer.WordOutput with .substitutions, .deletions, .insertions, .hits.

    '''
    return jiwer.process_words(reference, hypothesis)


# Evaluate on LibriSpeech single-speaker sample
hypothesis_text = ' '.join(
    c['text'].strip().lower()
    for c in word_result['chunks']
    if c.get('text')
)

wer_score = compute_wer(REFERENCE_TEXT, hypothesis_text)
details   = compute_detailed_errors(REFERENCE_TEXT, hypothesis_text)

print(f'Reference ({len(REFERENCE_TEXT.split())} words):\n  {REFERENCE_TEXT}\n')
print(f'Hypothesis:\n  {hypothesis_text}\n')
print(f'WER           : {wer_score:.4f}  ({wer_score*100:.2f} %)')
print(f'Substitutions : {details.substitutions}')
print(f'Deletions     : {details.deletions}')
print(f'Insertions    : {details.insertions}')
print(f'Hits          : {details.hits}')

### Exercise 7.2 — Model Size vs. Accuracy

The cell below evaluates three Whisper models on the same LibriSpeech sample.  Run it, then fill in the table.

| Model | WER (%) | Inference time (s) |
|---|---|---|
| `whisper-tiny` | | |
| `whisper-base` | | |
| `whisper-large-v3-turbo` | | |

In [ ]:
import time

models_to_compare = [
    'openai/whisper-tiny',
    'openai/whisper-base',
    'openai/whisper-large-v3-turbo',
]
comparison = []

for model_id in models_to_compare:
    print(f'\nEvaluating {model_id} ...')
    pipe = load_asr_pipeline(model_id, device, torch_dtype)

    t0      = time.time()
    res     = transcribe(pipe, SINGLE_AUDIO, SINGLE_SR, return_timestamps=True)
    elapsed = time.time() - t0

    hyp = res['text'].strip().lower()
    wer = compute_wer(REFERENCE_TEXT, hyp)
    comparison.append({'model': model_id.split('/')[-1], 'wer': wer, 'time_s': elapsed})
    print(f'  WER = {wer:.4f}   time = {elapsed:.1f} s')

    del pipe
    if device == 'cuda':
        torch.cuda.empty_cache()

print()
print(f'{"Model":<28}  {"WER (%)":>9}  {"Time (s)":>10}')
print('─' * 52)
for r in comparison:
    print(f'{r["model"]:<28}  {r["wer"]*100:>9.2f}  {r["time_s"]:>10.1f}')

> **Q7.1** (a) What is the WER gap between `whisper-tiny` and `whisper-large-v3-turbo`?  Is the accuracy gain worth the extra inference time for a real-time application?  (b) Look at the error breakdown in Exercise 7.1 — which error type dominates (S, D, or I)?  What kinds of words (proper nouns, function words, rare words) do you expect to contribute most?

> **Q7.2** Suppose you want to develop an altenartive to WER that considers the most important information, e.g., symptoms in a patient-agent interactions, What information would you use to weight the errors?



### Stretch — Diarization Error Rate (DER)

$$\text{DER} = \frac{\text{missed speech} + \text{false alarm} + \text{speaker confusion}}{\text{total reference speech time}}$$

Because we built `MULTI_AUDIO` ourselves from LibriSpeech, we have **exact ground-truth boundaries** in `GROUND_TRUTH`.  Run the cell below to compute DER directly — no manual annotation needed.

In [ ]:
# Stretch — DER using exact LibriSpeech ground truth

from pyannote.metrics.diarization import DiarizationErrorRate

reference = Annotation()
for gt in GROUND_TRUTH:
    reference[Segment(gt['start'], gt['end'])] = gt['speaker']

metric = DiarizationErrorRate()
der    = metric(reference, annotation)
detail = metric(reference, annotation, detailed=True)

print(f'DER               : {der:.4f}  ({der*100:.2f} %)')
print(f'Missed speech     : {detail["missed detection"]:.4f}')
print(f'False alarm       : {detail["false alarm"]:.4f}')
print(f'Speaker confusion : {detail["confusion"]:.4f}')
print()
print('Note: any remaining DER on clean LibriSpeech speech is mainly')
print('due to the silence gaps being (mis)assigned to a speaker.')

---
## Section 8 — Failure Analysis

Understanding *when* and *why* systems fail is as important as knowing how to run them.

| Condition | ASR impact | Diarization impact |
|---|---|---|
| Overlapping speech | Transcribes one speaker, drops the other | Speaker confusion spikes |
| Background music/noise | Hallucinated words, high WER | VAD may suppress speech |
| Non-native accent | Higher substitution rate | Embeddings may cluster poorly |
| Very short turns (<1 s) | Usually fine | Segments often missed by VAD |
| Whispered speech | Very low confidence, high WER | Not detected by VAD |
| Telephone audio (8 kHz) | Moderate degradation | Embedding quality degrades |

### Exercise 8.1 — Noise Robustness

The cell below adds controllable white noise to `SINGLE_AUDIO` and measures WER degradation.  Experiment with `NOISE_LEVEL` and record your observations.

After completing the baseline, choose **one** additional failure mode from the table above and design your own experiment.

In [ ]:
# Experiment 1: Additive white noise
NOISE_LEVEL = 0.05   # try 0.01, 0.05, 0.10, 0.20, 0.50

noisy_audio = SINGLE_AUDIO + NOISE_LEVEL * np.random.randn(len(SINGLE_AUDIO)).astype(np.float32)
noisy_audio = np.clip(noisy_audio, -1.0, 1.0)

print('Clean audio:'); display(Audio(SINGLE_AUDIO, rate=SINGLE_SR))
print('Noisy audio:'); display(Audio(noisy_audio,  rate=SINGLE_SR))

noisy_result = transcribe(asr_pipe, noisy_audio, SINGLE_SR, return_timestamps=True)
noisy_hyp    = noisy_result['text'].strip().lower()
wer_clean    = compute_wer(REFERENCE_TEXT, hypothesis_text)
wer_noisy    = compute_wer(REFERENCE_TEXT, noisy_hyp)

print(f'\nNoise level    : {NOISE_LEVEL}')
print(f'WER (clean)    : {wer_clean:.4f}  ({wer_clean*100:.2f} %)')
print(f'WER (noisy)    : {wer_noisy:.4f}  ({wer_noisy*100:.2f} %)')
print(f'WER increase   : {(wer_noisy - wer_clean)*100:+.2f} pp')
print(f'\nNoisy transcript: {noisy_hyp}')

# Document observations:
# NOISE_LEVEL tested  :
# WER increase        :
# Types of errors     :
# Suspected mechanism :

In [ ]:
# Experiment 2 (open-ended) -- Design your own failure case
# Choose a condition from the table above.
#
# Example solution: telephone bandwidth simulation (low-pass at 4 kHz)
#
# YOUR CODE HERE

> **Q8.1** For the white noise experiment: at what noise level does WER first exceed 20%?  What types of errors (substitutions, deletions, insertions) increase the most?  Why does additive Gaussian noise affect ASR accuracy?

> **Q8.2** Describe the additional failure mode you tested (Experiment 2).  What specific errors appeared?  Propose one concrete technique — a pre-processing step, model variant, or training strategy — that would make the pipeline more robust to that condition.